# 3.1 資料與時點處理

## 資料範圍與時點控制

**母體**　S&P 500 成分股，Tiingo 日頻價格，2000-01 至 2025-12，共 **6,287 個交易日**
**基本面**　SEC EDGAR XBRL companyfacts　**產業**　GICS 十一大類

### 三處時點（point-in-time）控制

| 環節 | 控制方式 | 若不控制的後果 |
| :--- | :--- | :--- |
| **成分股認定** | 依 `index_memberships` 的納入／剔除日，每期僅取**當時真實在指數內**者；含 170 檔已下市股 | 存活者偏誤 |
| **基本面對齊** | 一律取 `filed ≤ 形成期結束日` 的最新一筆，**非**財報期末日 | 使用尚未公開的資訊 |
| **特徵計算窗** | 僅用形成期窗內資料，交易期價格不參與任何估計 | 前視偏誤 |

資料庫共 **843 檔**成分股。

## 滾動回測設計

| 參數 | 設定 |
| :--- | :---: |
| 形成期長度 | **252** 個交易日（約一年） |
| 交易期長度 | **126** 個交易日（約半年） |
| 滾動步長 | **21** 個交易日（約一月） |
| 同時重疊期數 | **6**（= 126 / 21） |

::: {.callout-important}

### 一項對統計設計有決定性影響的性質

滾動步長 **小於** 交易期長度 → 任一時點有 **6 個交易期同時運行**
→ **逐期報酬序列存在結構性自相關**。

此性質決定了 §3.5 為何不能以「期」為抽樣單位，
以及為何主檢定必須使用能保住自相關結構的 **block bootstrap**。

:::

**交易成本**　單邊 **0.29%**（Do & Faff, 2012 之 ~30bps 估計），
進出場各扣一次 → 一往返約名目額 **0.58%**。
該估計樣本期為 1962–2009，套用於 2000–2025 **偏保守**；
第四章另報 break-even 成本。

# 3.2 形成期的四層架構

## 設計原理：讓單變因成為結構保證

配對交易的形成期通常被實作為**單一整體流程** →
更換任一環節時難以歸因其效果。

本研究拆解為四個**可獨立替換**的層：

```
特徵萃取  →  分組  →  群內排序  →  統計篩選
```

各層以標準化介面銜接：
特徵層輸出 $(N 	imes d)$ 矩陣 → 分組層輸出 $\{股票 	o 組標籤\}$
→ 排序層在組內選前 $N$ 組 → 篩選層對價差施加統計檢定。

（上圖為**介面**的相依順序；排序與篩選在實作上的**施行**次序依後端而異，見 §3.2 排序層與篩選層。）

::: {.callout-tip}

此架構使「單變因對照」成為**結構上的保證**，而非人為約定——
檢定分組方法時，其餘三層的參數完全相同。

:::

## 特徵層：19 維（連續 7 維）

| 區塊 | 維度 | 內容 | 依據 |
| :--- | :---: | :--- | :--- |
| 報酬主成分載荷 | **5** | 形成期日報酬 PCA 前 5 主成分載荷，以特徵值平方根加權 | Avellaneda & Lee (2010) |
| 公司基本面 | **2** | 對數市值、盈餘殖利率（1/PE） | — |
| GICS 產業 one-hot | **12** | 11 大產業 + 1 未知 | — |

各區塊**獨立標準化**後依權重拼接（避免 one-hot 欄位數稀釋連續特徵的距離量測）。

缺失值以**產業中位數**插補後 winsorize（1%/99%）。

> ⚠️ 第四章將指出：此插補方式構成一條**未被察覺的產業資訊管道**，
> 並引入全域中位數插補作為對照。

## 分組層：三種分群 + 對照組

| 方法 | 關鍵參數 | 群數決定方式 |
| :--- | :--- | :--- |
| **HDBSCAN** | `min_cluster_size`=5, `min_samples`=2 | 資料驅動，可標記噪音 |
| **Agglomerative** | average linkage，門檻取距離分布 **75 分位** | 由門檻決定 |
| **K-means** | $k$ = **同期 Agglomerative 的群數** | 對齊使量級可比 |
| **GICS（對照）** | — | 11 大產業，不跑分群 |

::: {.callout-note}

### K-means 的群數為何要對齊

K-means 需**預先指定**群數，若任意給定將使其與其他方法不可比。
本研究先跑一次 Agglomerative 取得資料驅動的群數再餵給 K-means，
確保比較聚焦於**分群機制**而非**粒度**。

:::

HDBSCAN 的噪音點（標籤 −1）與過小群（成員 < 5）併入「Unknown」，於排序層跳過。

## 排序層與篩選層

### 排序層：三種距離準則

| 準則 | 定義 |
| :--- | :--- |
| **SSD** | 正規化對數價格路徑的平方差總和（GGR, 2006） |
| **DTW** | 動態時間校正，Sakoe-Chiba 頻帶寬 15（許鈞翔, 2025） |
| **SSD-DTW-PCA** | 兩距離的主成分融合，取第一主成分 |

### 篩選層：三道檢定（任一未過即淘汰）

1. **ADF 共整合**　殘差 $p < 0.05$（Engle & Granger, 1987）
2. **OU 半衰期**　$HL = -\ln 2 / \lambda$，要求 $1 \le HL \le 42$ 日
3. **Hurst 指數**　$H < 0.5$（均值回歸傾向）

半衰期上限 42 日 = 交易期 126 日的 1/3，確保價差有足夠時間回歸。

::: {.callout-important}

### 排序與篩選的施行次序依後端而異

| 後端 | 實際流程 |
| :--- | :--- |
| **SSD** | 先按 SSD 排序 → 取前 $\max(200,\ 	ext{Top}N	imes15)$ 為候選 → **依距離順序**逐一檢定 → 湊滿 $	ext{Top}N	imes5$ 即停 → 再排序取前 $N$ |
| **DTW／SSD-DTW-PCA** | 群內**全配對**逐一檢定（無候選上限）→ 才排序取前 $N$ |

兩者的**共同**後果相同，且是理解第四章結果的關鍵：
**候選池不足時，系統會被迫接受距離更遠的配對**——SSD 因候選上限與提前中止而更早觸發，
DTW 則因通過篩選者不足 $N$ 組而觸發。

:::

::: {.callout-note}

### 這是「可獨立替換」的一項例外，須明確揭露

四層架構使分組層的替換成為乾淨的單變因對照。但**排序層的替換同時改變了
排序與篩選的施行次序**（上表），故排序層之間的比較嚴格說並非單一變因。

本研究的主命題落在**分組層**（命題 1）與**交易端**（命題 2），
兩者皆固定排序後端，不受此影響；涉及排序層的比較僅作描述性報告。

:::

# 3.3 「動態分群」的界定

標題所稱**動態分群**＝分群模型於**每一形成期獨立重新配適**，
而非全樣本分群一次後固定。

| 性質 | 內容 |
| :--- | :--- |
| **逐期重估** | 每期以該期 252 日視窗重建特徵、重新配適；不保留跨期狀態 |
| **重估次數** | **295 期**（2000-01-03 → 2024-07-01），三種分群法各配適 295 次 |
| **群數隨資料變動** | HDBSCAN 由密度決定；Agglomerative 取當期距離 75 分位；K-means 對齊之 |
| **前視偏誤防範** | 全樣本分群會讓 2005 年的搜尋空間受 2020 年共變影響 |

## 動態性的實測①：配對層級不具鑑別力

| 分組方法 | 相鄰期配對重疊率 | 配對存續期數（中位） |
| :--- | ---: | ---: |
| Agglomerative | 9.8% | 1 |
| HDBSCAN | 9.4% | 1 |
| K-means | 6.4% | 1 |
| **GICS（靜態對照）** | **11.7%** | 1 |

::: {.callout-warning}

### 靜態的 GICS 週轉率反而最高

配對週轉主要由**排序層與篩選層**驅動——即使分組固定，
每期的距離排序與共整合檢定結果本就不同。
**此指標無法佐證分群層的動態性。**

:::

## 動態性的實測②：分群層級以 ARI 量測

重跑形成期前兩層取出群標籤，計算相鄰期在**共同標的**上的
**調整蘭德指數**（群編號無意義，故比對「兩兩是否同群」；ARI 已對隨機一致校正）。

| 分組方法 | 群數 | 相隔 1 期（21 日） | 相隔 6 期（126 日） |
| :--- | ---: | ---: | ---: |
| Agglomerative | 104 | **0.709** | 0.483 |
| HDBSCAN | 16 | **0.685** | 0.498 |
| K-means | 104 | **0.515** | 0.350 |
| **GICS（靜態）** | 11 | **1.000** | 1.000 |

- **結構確實逐期改變**：相隔 21 日已有三至五成分群關係改變
- **改變隨時間累積**：一個交易期走完，過半結構已不同
- **K-means 最不穩定**（隨機初始化；Agglomerative 的階層合併較不敏感）

> **範圍聲明**：確立分群**確實在變**，但**未**檢定這種改變是否**有益**——
> 本研究無「靜態分群」對照組。逐期重估是前視偏誤防範的必要設計，
> 而非受檢定的處理。

## 消融矩陣：4 分組 × 3 排序

## 4 分組 × 3 排序消融矩陣

|  | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（對照）** | ✓ | ✓ | ✓ |
| HDBSCAN | ✓ | ✓ | ✓ |
| Agglomerative | ✓ | ✓ | ✓ |
| K-means | ✓ | ✓ | ✓ |

固定特徵層、篩選層與交易端 → **唯一變因為分組方法**。
每格再展開 `top_n` × `stop_loss`（5 × 3 = 15 種配置）。

同一排序準則下的 ML 分群與 GICS 構成直接對照，共 **9 組**（3 分群 × 3 排序）。

### 三項受控消融（用於失敗歸因）

| 消融 | 參數 | 檢驗什麼 |
| :--- | :--- | :--- |
| **產業先驗強度** | `sector_onehot_weight` ∈ {1.0, 0} | one-hot 對跨產業配對的距離懲罰 |
| **統計篩選** | `filter_mode` ∈ {coint, none} | 篩選的貢獻，及其與分群的**交互作用** |
| **分組維度零點** | `cluster_method` = **none** | 「限制搜尋空間」本身的價值 |

> 若無「不分組」對照，所比較的僅是不同的**限制方式**，而非限制本身。

# 3.4 門檻選擇的深度學習實作（命題 2）

## 動作空間：門檻選擇式而非逐日定位

**基準**　Z-Score 規則：$|z| > 2.0$ 進場、$z$ 穿越 0 平倉、期末強平、另設停損。

> **動作選單（9 個）**：SKIP（不交易）
> ＋ 8 組 $(entry\_z,\ exit\_z) \in \{1.5, 2.0, 2.5, 3.0\} 	imes \{0.0, 0.5\}$

代理人對每組配對、每期**僅做一次決策**，選定後交由標準 Z-Score 狀態機執行整期。

### 三項結構性保證

1. 選單含靜態基準 $(2.0, 0.0)$ → **策略空間必然包含 Z-Score 基準**
2. 訓練樣本不足時自動選基準動作 → 樣本初期行為**等同**基準
3. **SKIP** 使代理人可拒絕交易 —— 固定規則不具備的選擇性

::: {.callout-note}

### 為何不用逐日定位的動作空間

本研究早期版本採「每日自由決定持倉」，結果模型對**日級噪音計時**，
換手成本大幅上升、績效顯著劣於基準。
失敗根因被隔離於**動作空間設計**，而非學習演算法本身。

:::

## 狀態空間與學習問題的性質

**狀態 = 12 維形成期特徵**（全部可於交易期開始前計算）

| 類別 | 特徵 |
| :--- | :--- |
| 偏離狀態 | 期末 $z$、期末 $\|z\|$ |
| 回歸品質 | 零穿越頻率、OU 半衰期對數 |
| 近期 regime | 近 21 日 $z$ 波動相對全期、近 21 日 $z$ 趨勢 |
| 配對性質 | 兩檔報酬相關係數、波動比、對沖比例偏離 1 的幅度 |
| 可交易性 | 價差振幅、形成期 $\|z\|>2$ 佔比、形成期最大 $\|z\|$ |

::: {.callout-tip}

### 這是全資訊監督回歸，不是 bandit

歷史配對期中，**全部 9 個動作的報酬都可精確反事實回算**
（對該期價格逐一模擬 9 組門檻）→ **無探索問題、樣本效率最高**。

網路：MLP（12 → 隱藏 64 → 9 個動作報酬），MSE 損失，每期增量訓練 40 epoch。

:::

::: {.callout-warning}

### 命名說明

本研究早期版本及程式碼中稱此交易端為「**DRL**」（deep reinforcement learning）。
依上述，其學習問題為**全資訊監督回歸**而非強化學習——動作報酬可完整反事實回算，
不存在探索與利用的取捨。為避免與真正的部分回饋設定混淆，
本文一律稱其為 **DL-THR**（deep-learning threshold selection）。

此區分具實質意義：下一張的受控對照刻意實作了一個**真正的** contextual bandit
版本（**RL-THR**），若主臂仍沿用「DRL」之名，該節將無法閱讀。

> `result.db` 的 strategy id（如 `Grid (AGG-SSD-DRL)`）為**歷史命名**，
> 未隨本文更動，以維持與既有回測輸出的可對照性。

:::

## 前視偏誤的防範與隨機性處理

::: {.callout-important}

### walk-forward 增量訓練

> 期 $k$ 的決策，**僅使用「交易期已於期 $k$ 開始前結束」的樣本**訓練。

實作為對訓練緩衝區施加 `trade_end < trade_start_k` 過濾。
可用樣本 < 200 筆時，自動選用基準動作 $(2.0, 0.0)$。

:::

**隨機性處理**　網路未固定隨機種子 → 對三種 ML 配對底各執行**五輪獨立重訓**，
以中位數與全距報告，避免單次訓練的隨機性被誤讀為方法效果。

## 受控對照：把全資訊標籤換成部分回饋

如前所述，**DL-THR 不是強化學習**：9 個動作的報酬在每個歷史配對期上都能精確
反事實回算，模型拿到的是完整答案卷，學習形式為全資訊監督回歸
（MLP + MSE，決策取 $\arg\max$），既無探索問題、亦無序列信用分配。

為分離「學習演算法」與「這個問題碰巧是全資訊的」兩件事，本研究另建 **RL-THR**
作為受控對照。它保持動作選單、12 維狀態、網路結構與 walk-forward 切分**逐位元相同**
（特徵函式直接引用同一份實作），只改兩處：

| 環節 | DL-THR | RL-THR |
| :--- | :--- | :--- |
| 訓練標籤 | 9 個動作全部反事實回算 | **只有實際選中的那一個** |
| 損失 | 全 9 維 MSE | **遮罩 MSE**（僅選中動作的輸出單元收梯度） |
| 決策 | 恆 $\arg\max$ | **$\varepsilon$-greedy** |

於是 RL-THR 成為真正的部分回饋問題，兩者相減即為**反事實標籤的價值**。

::: {.callout-important}

### 形式歸屬：contextual bandit，沒有 $\gamma$

RL-THR 是**情境式拉霸機**而非序貫 MDP。每組配對每期只做一次決策，
且 12 維狀態全部由形成期視窗算出——選哪個門檻**不會改變下一期的狀態**。
沒有狀態轉移就沒有東西可以 bootstrap，硬加折扣因子只是裝飾。
其「RL 性」來自部分回饋與探索／利用權衡（Sutton & Barto, 2018, 第 2 章），
而非 TD 學習。本研究不宣稱序貫決策。

真正的序貫版本（逐日定位、$\gamma$=0.99、bootstrapped target）已於三代實作中
系統性證偽，見 §3.4 註與封存紀錄。

:::

::: {.callout-warning}

### RL-THR 少一項結構性保證，且兩個效應無法分離

DL-THR 保證「樣本不足時退回基準 → 暖身期 $\equiv$ Z-Score」。RL-THR **無法**如此：
若暖身期一律選基準，其餘八個動作永遠沒有樣本。探索必須從第一期開始。

因此兩臂的差距同時包含**資訊量**（每期 1 筆觀測 vs 9 筆）與**探索成本**
（$\varepsilon$ 抽中時實際下單、虧損計入績效）。本設計無法拆解為兩個獨立數字——
不探索就沒有樣本，這是部分回饋的定義。掃三組 $\varepsilon$
（$0.05$、$0.10$、$0.20\!\to\!0.02$ 衰減）以界定取捨曲線，
並以**對 bandit 最有利者**作結論，使推論保守。

:::

# 3.5 統計檢定方法

## 抽樣單位：為何不能用參數網格

本研究早期版本以「參數網格」為抽樣單位——對 15 種 `top_n` × `stop_loss`
配置作配對 $t$ 檢定。

::: {.callout-important}

### 偽重複（pseudo-replication）

15 個「觀測」共用**同一份資料、同一段期間、同一批配對**：

- `top_n`=10 與 `top_n`=20 **共用 10 組配對**
- 三種停損是**同一批交易**的不同出場規則

觀測間高度相關 → 不滿足 $t$ 檢定的獨立性假設 →
**有效樣本數接近一條回測路徑，而非 15**。

:::

**本研究改以「時間」為抽樣單位。**

## 主檢定：逐日報酬差 + 循環 block bootstrap

$$\Delta r_t = r_{\text{處理},t} - r_{\text{對照},t}, \quad t = 1, \dots, 6287$$

檢定 $H_0: E[\Delta r] = 0$。6 期重疊部位使 $\Delta r_t$ 自相關，
故以循環 block bootstrap（Künsch 1989；Politis & Romano 1992）處理：

1. 首尾相接成環 → 抽長度 $L$ 的區塊接成等長新序列 → 重複 **10,000** 次
2. 平移該分布使中心為零（施加 $H_0$）→ 雙尾 $p$
3. **未平移**的分布取 2.5／97.5 百分位 → **95% 信賴區間**

**未持倉日記為 0**，非遺漏值——該日確實沒有部位；
視為遺漏將系統性排除策略「不交易」的行為。

**$L = 126$＝一個完整交易期**。區塊內部的持倉相關性完整保留，
區塊之間才允許被打散。此值由 `FORWARD_DAYS` 直接決定，非搜尋而得，
故**不作 $L$ 敏感度分析**。

::: {.callout-note}

### 為何不採 Newey-West HAC

兩法全程並行計算，**14 組對照結論完全一致**（命題 2 五組皆顯著、命題 1 九組皆不顯著）。
換掉的理由不在結果而在**假設與自由度**：

| | HAC | block bootstrap |
| :--- | :--- | :--- |
| 分布假設 | 漸近常態 | 無 |
| 額外自由度 | **落後階**（須另跑多組證明穩健） | 無（$L$ 由交易期長度給定） |
| 日報酬厚尾偏態 | 近似可能失準 | 不受影響 |

HAC 保留為各表的**對照欄**。

:::

## 信賴區間、多重檢定與報告口徑

### 信賴區間取代非劣性檢定

「無顯著差異」**不等於**「兩者相當」。區分兩者只需看**區間寬度**：

- 區間涵蓋 0 且**兩端皆小** → 效果縱使存在也不具實質意義
- 區間涵蓋 0 但**兩端皆大** → **檢定力不足**，資料無法區分

命題 1 即屬後者。舊版以非劣性檢定處理同一問題，須事前指定並辯護
容忍邊界 $\delta$；CI 承載相同資訊且不需此判斷，故改採 CI。

### 多重檢定校正

同一命題涉及多組對照時（如命題 1 的 9 組），
以 **Benjamini-Hochberg** 控制 FDR，同時報告原始與校正後 $p$ 值。

::: {.callout-important}

### 報告口徑：一律全網格等權組合

一切績效主張與統計檢定，皆以該策略 **15 個參數配置的等權組合**為口徑。

改報「網格最佳格」會內含 15 選 1 的選擇偏誤，必須再以 Deflated Sharpe 之類的
程序扣回去——而該程序的關鍵參數（試驗數 $N$）是判斷而非事實。
**等權組合沒有東西可挑，選擇偏誤自源頭消失。**

代價是絕對績效數字低於最佳格。這正是誠實的代價：
研究者事前並不知道哪一格會勝出。

*例外*：§4.6.1 regime 分層與 §4.6.2 break-even 成本表以最佳格計算——
二者刻畫風險的**形狀**而非主張績效水準，且最佳格對其論證方向是保守的。

:::

## 本章小結

::: {.callout-important}

本研究的方法設計圍繞一項原則：

> **任兩個待比較的策略之間，僅存在單一變因。**

- **形成期**：四層架構使此原則成為**結構上的保證**
- **交易期**：利用「可在同一批配對上施行」的性質達成同樣效果

統計方面，以**時間**為抽樣單位取代參數網格，
以 **block bootstrap** 為單一主檢定（同時給出 $p$ 值與信賴區間），
並一律以**全網格等權組合**為報告口徑——
使推論不依賴分布假設，也不留下「挑最佳配置」的空間。

:::